<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Goal:** Turn the validated model output into a prioritized human-review queue

We need:
- ranked pages
- priority/score
- reason code(s)
- recommended action
- short explanation of why the page was prioritized

The key is that the queue says “review this page first”, not “do this automatically.”

In [1]:
# Load data
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

target = "is_declining_label"

print("Rows:", len(df))
print("Features:", features)

Rows: 30000
Features: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']


In [2]:
# Action queue
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

X = df[features].copy()
y = df[target].copy()

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

queue_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

queue_model.fit(X_imputed, y)

df["model_score"] = queue_model.predict_proba(X_imputed)[:, 1]

print(df["model_score"].describe())

count    30000.000000
mean         0.541617
std          0.367347
min          0.000000
25%          0.170000
50%          0.740000
75%          0.890000
max          1.000000
Name: model_score, dtype: float64


In [3]:
# Reason code
def assign_reason_codes(row):
    reasons = []

    if row["content_age_days"] >= df["content_age_days"].quantile(0.75):
        reasons.append("CONTENT_AGING")

    if row["days_since_last_update"] >= df["days_since_last_update"].quantile(0.75):
        reasons.append("LONG_TIME_SINCE_UPDATE")

    if row["ctr"] <= df["ctr"].quantile(0.25):
        reasons.append("LOW_CTR")

    if row["avg_position"] >= df["avg_position"].quantile(0.75):
        reasons.append("WEAKER_AVG_POSITION")

    if row["impressions_90d"] <= df["impressions_90d"].quantile(0.25):
        reasons.append("LOW_RECENT_IMPRESSIONS")

    if not reasons:
        reasons.append("MODEL_PRIORITY")

    return ", ".join(reasons)


df["reason_code"] = df.apply(assign_reason_codes, axis=1)

In [4]:
# Priority and Recommended action
df["priority"] = pd.cut(
    df["model_score"],
    bins=[-np.inf, 0.50, 0.75, np.inf],
    labels=["Lower", "Medium", "High"]
)

def recommend_action(reason_code):
    if "CONTENT_AGING" in reason_code or "LONG_TIME_SINCE_UPDATE" in reason_code:
        return "Review whether a content refresh is warranted"

    if "LOW_CTR" in reason_code:
        return "Review title, snippet, and search-intent alignment"

    if "WEAKER_AVG_POSITION" in reason_code:
        return "Review relevance, structure, and search-intent alignment"

    if "LOW_RECENT_IMPRESSIONS" in reason_code:
        return "Review current visibility and whether the page remains strategically relevant"

    return "Conduct general content review"


df["recommended_action"] = df["reason_code"].apply(recommend_action)

In [5]:
# Ranked queue
action_queue = (
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "model_score",
            "priority",
            "reason_code",
            "recommended_action"
        ]
    ]
    .sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)

action_queue.insert(0, "rank", action_queue.index + 1)

action_queue.head(20)

,rank,content_id,client_id,content_type,model_score,priority,reason_code,recommended_action
0,1,content_0c0dc1068177,client_6208ef0f77,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted
1,2,content_c092ace0d33a,client_3fdba35f04,keyword article,1.0,High,LOW_CTR,"Review title, snippet, and search-intent align..."
2,3,content_7d1dcfd354f5,client_7f2253d7e2,keyword article,1.0,High,WEAKER_AVG_POSITION,"Review relevance, structure, and search-intent..."
3,4,content_0daf885ddaa7,client_7f2253d7e2,keyword article,1.0,High,LOW_CTR,"Review title, snippet, and search-intent align..."
4,5,content_7602d27a3586,client_6208ef0f77,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted
5,6,content_96ec5cfee15e,client_6208ef0f77,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted
6,7,content_66868d96318e,client_19581e27de,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted
7,8,content_b01a3c1455db,client_7f2253d7e2,keyword article,1.0,High,WEAKER_AVG_POSITION,"Review relevance, structure, and search-intent..."
8,9,content_1770f526befb,client_19581e27de,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted
9,10,content_8e32f425a490,client_9400f1b21c,keyword article,1.0,High,CONTENT_AGING,Review whether a content refresh is warranted


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The playbook is intended to **prioritize human review of pages showing signals associated with decline**; it is not intended to automatically decide which content should be changed or predict the effect of a specific intervention.

The **intended use** is to help content teams **prioritize which pages deserve review** and use model signals to make the **review process more systematic**. And some **limits** are that it ranks patterns; it does **not establish causality** and high-ranked pages is **not guaranteed to improve after an intervention**

In [6]:
print("Queue size:", len(action_queue))
print("High-priority pages:", (action_queue["priority"] == "High").sum())
print("Medium-priority pages:", (action_queue["priority"] == "Medium").sum())
print("Lower-priority pages:", (action_queue["priority"] == "Lower").sum())

print("\nValidation reference:")
print("5-fold mean Precision@50: 0.768")
print("5-fold std Precision@50: 0.086")

Queue size: 30000
High-priority pages: 14637
Medium-priority pages: 1604
Lower-priority pages: 13759

Validation reference:
5-fold mean Precision@50: 0.768
5-fold std Precision@50: 0.086


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on a recommendation, a person should check things the **model cannot know reliably**, such as:

Is the page still strategically important?
Is the information current?
Has search intent changed?
Does the proposed action actually make sense for this page?

**Never automate**:
publishing or rewriting content without review
deleting pages
redirects/canonical changes
making sensitive claims
overriding editorial judgment solely because of the model score

In [7]:
# human-review flag
action_queue["human_review_required"] = True

action_queue[
    ["rank", "content_id", "priority", "reason_code",
     "recommended_action", "human_review_required"]
].head(10)

,rank,content_id,priority,reason_code,recommended_action,human_review_required
0,1,content_0c0dc1068177,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
1,2,content_c092ace0d33a,High,LOW_CTR,"Review title, snippet, and search-intent align...",True
2,3,content_7d1dcfd354f5,High,WEAKER_AVG_POSITION,"Review relevance, structure, and search-intent...",True
3,4,content_0daf885ddaa7,High,LOW_CTR,"Review title, snippet, and search-intent align...",True
4,5,content_7602d27a3586,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
5,6,content_96ec5cfee15e,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
6,7,content_66868d96318e,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
7,8,content_b01a3c1455db,High,WEAKER_AVG_POSITION,"Review relevance, structure, and search-intent...",True
8,9,content_1770f526befb,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
9,10,content_8e32f425a490,High,CONTENT_AGING,Review whether a content refresh is warranted,True


In [8]:
assert action_queue["human_review_required"].all()

print(
    "Human review required for all queued pages:",
    action_queue["human_review_required"].all()
)

Human review required for all queued pages: True


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitor**:
validation performance over time,
Precision@K / ranking quality,
proportion of pages receiving each reason code and
whether the queue still looks useful to reviewers

Consider **revalidation when**:
model performance drops materially,
the content portfolio changes substantially,
search behavior changes,
new content types/archetypes become common and
the model's recommendations repeatedly conflict with expert review

In [9]:
# Baseline distributions
monitoring_baseline = {
    "mean_model_score": df["model_score"].mean(),
    "median_model_score": df["model_score"].median(),
    "mean_impressions_90d": df["impressions_90d"].mean(),
    "mean_ctr": df["ctr"].mean(),
    "mean_avg_position": df["avg_position"].mean(),
    "mean_content_age_days": df["content_age_days"].mean(),
    "mean_days_since_last_update": df["days_since_last_update"].mean()
}

monitoring_baseline

{'mean_model_score': np.float64(0.5416169175996037),
 'median_model_score': 0.74,
 'mean_impressions_90d': np.float64(5200.3663),
 'mean_ctr': np.float64(0.5107333333333334),
 'mean_avg_position': np.float64(16.342380000000002),
 'mean_content_age_days': np.float64(256.1678),
 'mean_days_since_last_update': np.float64(46.0983)}

In [10]:
reason_distribution = (
    action_queue["reason_code"]
    .value_counts()
    .head(10)
)

reason_distribution

,count
reason_code,
MODEL_PRIORITY,5481
"LOW_CTR, LOW_RECENT_IMPRESSIONS",4103
LONG_TIME_SINCE_UPDATE,3927
CONTENT_AGING,2802
LOW_CTR,1637
"LONG_TIME_SINCE_UPDATE, WEAKER_AVG_POSITION",1428
WEAKER_AVG_POSITION,1141
"CONTENT_AGING, LOW_CTR, WEAKER_AVG_POSITION",1135
"LONG_TIME_SINCE_UPDATE, LOW_CTR",1020


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
from pathlib import Path

output_dir = Path("../outputs")
figure_dir = Path("../figures")

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

In [12]:
queue_path = output_dir / "action_queue.csv"

action_queue.to_csv(queue_path, index=False)

print(f"Saved queue to: {queue_path}")
print("Rows exported:", len(action_queue))

Saved queue to: ../outputs/action_queue.csv
Rows exported: 30000


In [13]:
assert queue_path.exists()

exported_queue = pd.read_csv(queue_path)

print("Export verified.")
print("Shape:", exported_queue.shape)
exported_queue.head()

Export verified.
Shape: (30000, 9)


,rank,content_id,client_id,content_type,model_score,priority,reason_code,recommended_action,human_review_required
0,1,content_0c0dc1068177,client_6208ef0f77,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True
1,2,content_c092ace0d33a,client_3fdba35f04,keyword article,1.0,High,LOW_CTR,"Review title, snippet, and search-intent align...",True
2,3,content_7d1dcfd354f5,client_7f2253d7e2,keyword article,1.0,High,WEAKER_AVG_POSITION,"Review relevance, structure, and search-intent...",True
3,4,content_0daf885ddaa7,client_7f2253d7e2,keyword article,1.0,High,LOW_CTR,"Review title, snippet, and search-intent align...",True
4,5,content_7602d27a3586,client_6208ef0f77,keyword article,1.0,High,"LONG_TIME_SINCE_UPDATE, LOW_CTR",Review whether a content refresh is warranted,True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.